In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab12.ipynb")

# Lab 12: Modeling, Loss Functions, and Model Evaluation

Apply the modeling process from **Lecture 22** and the evaluation methods from **Lecture 23** to restaurant tips. 

- Calculate residuals, MSE, MAE, and RMSE.
- Fit simple linear regression analytically and check its estimating equations.
- Compare constant and linear models, investigate outliers, and evaluate log transformations.

Use NumPy and pandas. Keep full precision, use population standard deviations (`ddof=0`), and define residuals as observed minus predicted. All comparisons describe fit on the observed sample; they do not establish predictive performance on new data.

<a id="question-navigation"></a>
## Question Navigation

- [1: Residuals and Prediction Loss](#q1)
- [2: Analytical Least-Squares Fit](#q2)
- [3: Estimating Equations](#q3)
- [4: Evaluate the Linear Model](#q4)
- [5: Mean and Median Constant Models](#q5)
- [6: Constant versus Linear Models](#q6)
- [7: Loss Curves for Constant Models](#q7)
- [8: Outlier Sensitivity](#q8)
- [9: Residual Diagnostics](#q9)
- [10: Fit a Log-Response Model](#q10)

## Data and modeling process

Choose a model, choose a loss, fit its parameters, and evaluate the result. Here `total_bill` is the input and `tip` is the response, both measured in dollars. The setup loads Seaborn's tips dataset; internet access or a cached copy is needed on the first run.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Load Data

In [ ]:
tips = sns.load_dataset('tips').reset_index(drop=True)
x_total_bills = tips['total_bill'].to_numpy(dtype=float)
y_tips = tips['tip'].to_numpy(dtype=float)
tips.head()

In [ ]:
# Inspect the association before fitting a model.
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x_total_bills, y_tips, alpha=0.6)
ax.set(xlabel='Total bill ($)', ylabel='Tip ($)', title='Restaurant bills and tips')
plt.tight_layout()
plt.show()

---
<a id="q1"></a>
### Question 1: Residuals and Prediction Loss (12 points)

Evaluate a candidate line that predicts tip as `1.25 + 0.08 * total_bill`.

1. Calculate `candidate_predictions` for every bill in `x_total_bills`.
2. Calculate `candidate_residuals` as observed tips minus predicted tips.
3. Create `candidate_audit` with these columns, in order:
   - `observed`: values from `y_tips`.
   - `predicted`: the candidate predictions.
   - `residual`: observed minus predicted.
   - `squared_loss`: each residual squared.
   - `absolute_loss`: each residual's absolute value.
4. Set `signed_error` to the mean residual and `squared_error` to the mean squared residual.

Keep the original observation order and use a default integer index. Do not round. Positive and negative residuals can cancel, so signed error alone does not measure prediction accuracy.

In [ ]:
# Calculate predictions and residuals.
candidate_predictions = ...
candidate_residuals = ...

# Build the audit table in the required column order.
candidate_audit = pd.DataFrame({
    'observed': ...,
    'predicted': ...,
    'residual': ...,
    'squared_loss': ...,
    'absolute_loss': ...
})

# Calculate the mean signed error and mean squared error.
signed_error = ...
squared_error = ...
candidate_audit.head()

In [ ]:
grader.check("q1")

[↑ Question navigation](#question-navigation)

---
<a id="q2"></a>
### Question 2: Analytical Least-Squares Fit (12 points)

Fit a line predicting tip from total bill using the analytical formulas from Lecture 22.

1. Subtract each array's mean to obtain `x_centered` and `y_centered`.
2. Calculate `Sxx`, the sum of squared centered bill values.
3. Calculate `Sxy`, the sum of the elementwise products of centered bills and centered tips.
4. Calculate the slope `theta1_hat = Sxy / Sxx`.
5. Calculate `theta0_hat` so the fitted line passes through the sample means: intercept = mean tip minus slope times mean bill.
6. Store the fitted values for all observations in `linear_predictions`.

Use NumPy calculations rather than a regression library. Keep the original observation order and do not round.

In [ ]:
# Center the input and response arrays.
x_centered = ...
y_centered = ...

# Calculate the centered sums.
Sxx = ...
Sxy = ...

# Calculate slope, intercept, and fitted values.
theta1_hat = ...
theta0_hat = ...
linear_predictions = ...

In [ ]:
grader.check("q2")

[↑ Question navigation](#question-navigation)

---
<a id="q3"></a>
### Question 3: Estimating Equations (10 points)

Check the two estimating equations for the fitted least-squares line from Lecture 22.

1. Calculate `fitted_residuals` as `y_tips` minus `linear_predictions`.
2. Set `mean_residual` to the mean fitted residual.
3. Set `mean_x_residual` to the mean of the elementwise products of `x_total_bills` and `fitted_residuals`.
4. Set `normal_equations_hold` to a Boolean that is true only when **both** means are close to zero. Use `np.isclose` with `atol=1e-10` for each comparison.

These equations describe the fitted line's optimality. A mean residual near zero does not imply that individual prediction errors are small.

In [ ]:
# Calculate residuals and the two estimating-equation means.
fitted_residuals = ...
mean_residual = ...
mean_x_residual = ...

# Check that both means are close to zero.
normal_equations_hold = ...

In [ ]:
grader.check("q3")

[↑ Question navigation](#question-navigation)

---
<a id="q4"></a>
### Question 4: Evaluate the Linear Model (10 points)

Evaluate the fitted line using all observations, following Lecture 23.

1. Using `fitted_residuals`, calculate:
   - **MSE:** mean squared residual.
   - **MAE:** mean absolute residual.
   - **RMSE:** square root of MSE.
2. Store the three values in a pandas Series named `linear_metrics`, indexed `MSE`, `MAE`, `RMSE`, in that order.

MAE and RMSE are in dollars; MSE is in squared dollars. Do not round.

In [ ]:
# Calculate each metric using all fitted residuals.
linear_metrics = pd.Series({
    'MSE': ...,
    'MAE': ...,
    'RMSE': ...
})
linear_metrics

In [ ]:
grader.check("q4")

[↑ Question navigation](#question-navigation)

---
<a id="q5"></a>

### Question 5: Mean and Median Constant Models (10 points)

A **constant model** predicts the same tip for every observation.

1. Set `mean_constant` to the mean of `y_tips`. This constant minimizes MSE.
2. Set `median_constant` to the median of `y_tips`. This constant minimizes MAE.
3. For each constant model, calculate **MSE**, **MAE**, and **RMSE** using all observed tips. Define residuals as observed tip minus the constant prediction.
4. Store the results in a DataFrame named `constant_comparison`:
   - **Rows:** `Mean`, `Median`, in that order.
   - **Columns:** `MSE`, `MAE`, `RMSE`, in that order.

Do not round your results.

In [ ]:
mean_constant = ...
median_constant = ...

# Calculate residuals for each constant model.
mean_residuals = ...
median_residuals = ...

# Each list contains the metric for the mean model, then the median model.
constant_comparison = pd.DataFrame({
    'MSE': [..., ...],
    'MAE': [..., ...],
    'RMSE': [..., ...]
}, index=['Mean', 'Median'])
constant_comparison

In [ ]:
grader.check("q5")

[↑ Question navigation](#question-navigation)

---
<a id="q6"></a>
### Question 6: Constant versus Linear Models (10 points)

Compare two models using all observed tips:
- **Constant model:** predicts `mean_constant` for every observation.
- **Linear model:** uses the fitted values in `linear_predictions`.

1. Calculate `constant_mse` and `linear_mse` against `y_tips`.
2. Create a DataFrame named `model_comparison`:
   - **Rows:** `Constant`, `Linear`, in that order.
   - **Columns:** `MSE`, `RMSE`, in that order.
   - Calculate each RMSE as the square root of its MSE.
3. Set `rmse_reduction` to the constant model's RMSE minus the linear model's RMSE. A positive value means the linear model has lower RMSE.
4. Set `best_model` to the row label with the lowest MSE. Choose `Constant` if there is a tie.

Evaluate both models on the original tip scale. Do not round your results.

In [ ]:
# Calculate MSE for each model using all observed tips.
constant_mse = ...
linear_mse = ...

# Each list contains the constant model first, then the linear model.
model_comparison = pd.DataFrame({
    'MSE': [..., ...],
    'RMSE': [..., ...]
}, index=['Constant', 'Linear'])

# Calculate constant RMSE minus linear RMSE.
rmse_reduction = ...

# Select the row label with the lowest MSE.
best_model = ...
model_comparison

In [ ]:
grader.check("q6")

[↑ Question navigation](#question-navigation)

---
<a id="q7"></a>

### Question 7: Loss Curves for Constant Models (10 points)

Consider five constant models that predict tips of **[1., 2., 3., 4., 5.]** for every observation.

1. For each value in `constant_candidates`, calculate:
   - **MSE:** store the five results in the NumPy array `constant_mse`.
   - **MAE:** store the five results in the NumPy array `constant_mae`.
2. Keep both arrays in the same order as `constant_candidates`.
3. Set `best_mse_candidate` to the candidate tip value with the lowest MSE.
4. Set `best_mae_candidate` to the candidate tip value with the lowest MAE.

Use all values in `y_tips`. If multiple candidates share the minimum loss, choose the first one. Return the **candidate tip value**, not its index or loss.

These selections identify the best models among the five supplied candidates. Do not round your results.

In [ ]:
constant_candidates = np.array([1., 2., 3., 4., 5.])

# Calculate MSE and MAE for each candidate.
constant_mse = ...
constant_mae = ...

# Select the candidate value with the smallest loss.
best_mse_candidate = ...
best_mae_candidate = ...

In [ ]:
grader.check("q7")

[↑ Question navigation](#question-navigation)

In [ ]:
# Visualize the loss curves from Question 7.
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(constant_candidates, constant_mse, marker='o')
ax[1].plot(constant_candidates, constant_mae, marker='o')
for a, label in zip(ax, ['MSE ($²)', 'MAE ($)']):
    a.set(xlabel='Constant predicted tip ($)', ylabel=label)
plt.tight_layout()
plt.show()

---
<a id="q8"></a>

### Question 8: Outlier Sensitivity (8 points)

Investigate how an unusually large tip affects the mean and median.

1. Append **$100** to `y_tips`, storing the result in `tips_with_outlier`. Keep the original array unchanged.
2. Calculate `outlier_mean` and `outlier_median` from the new array.
3. Calculate `mean_change` and `median_change` as **new value minus original value**.
4. Set `more_sensitive` to `'Mean'`, `'Median'`, or `'Tie'` by comparing the absolute changes.

Do not round your results.

In [ ]:
tips_with_outlier = ...
outlier_mean = ...
outlier_median = ...

mean_change = ...
median_change = ...

# Compare the absolute changes.
more_sensitive = ...

In [ ]:
grader.check("q8")

[↑ Question navigation](#question-navigation)

---
<a id="q9"></a>

### Question 9: Residual Diagnostics (10 points)

Examine whether the fitted linear model makes systematic errors for different bill amounts.

1. Create `residual_diagnostics` with the following columns, in order:
   - `total_bill`: observed bill amounts.
   - `observed`: observed tips.
   - `predicted`: fitted linear predictions.
   - `residual`: observed tip minus predicted tip.

   Keep the original observation order and use a default integer index.

2. Calculate the mean residual for each bill group:
   - **Low:** `total_bill < 15`
   - **Middle:** `15 <= total_bill < 30`
   - **High:** `total_bill >= 30`

   All bill amounts are in dollars.

3. Store these means in a Series named `residual_bin_means`, indexed `Low`, `Middle`, `High`, in that order.

A positive mean residual indicates underprediction; a negative mean indicates overprediction. An overall mean residual near zero does not rule out patterns within groups.

In [ ]:
residual_diagnostics = pd.DataFrame({
    'total_bill': ...,
    'observed': ...,
    'predicted': ...,
    'residual': ...
})

# Define the three bill groups.
low_mask = ...
middle_mask = ...
high_mask = ...

# Calculate the mean residual in each group.
residual_bin_means = pd.Series([
    ...,
    ...,
    ...
], index=['Low', 'Middle', 'High'])

In [ ]:
grader.check("q9")

[↑ Question navigation](#question-navigation)

In [ ]:
# Inspect residual structure, as in Lecture 23.
fig, ax = plt.subplots(figsize=(6, 3))
ax.scatter(residual_diagnostics['total_bill'], residual_diagnostics['residual'], alpha=0.6)
ax.axhline(0, color='black', linestyle='--')
ax.set(xlabel='Total bill ($)', ylabel='Observed − predicted tip ($)', title='Residuals versus input')
plt.tight_layout()
plt.show()

---
<a id="q10"></a>
## Question 10: Fit a Log-Response Model (8 points)

All tips are positive. Create `log_tips = np.log(y_tips)` using the natural logarithm. Fit a line predicting log tip from total bill using the analytical SLR formulas. Store `log_slope`, `log_intercept`, `log_predictions`, and `log_residuals`. The residuals here are on the log scale.

In [ ]:
log_tips = ...
log_slope = ...
log_intercept = ...
log_predictions = ...
log_residuals = ...

In [ ]:
grader.check("q10")

[↑ Question navigation](#question-navigation)

## Submission

Run all cells in order, review the displayed test results, and save the notebook before running the generated export cell. Submit the exported ZIP file to Gradescope according to the course instructions. The instructor source contains solutions and hidden tests; distribute the student notebook generated by Otter Assign.

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)